# ML-08 — Capstone Modeling Lane



**Question:** Can pre-existing content, search, and engagement signals prioritize pages with an observed declining trend for review?



This notebook compares the Week-4 rule baseline with three classifiers on the same unseen-client holdout. It reports ranking metrics, model reliance, and concrete errors. The analysis is directional decision support; it does not estimate the causal effect of refreshing content.

## 1. Method choice and why



**Lane:** supervised binary classification used as a ranking problem. The observed label says whether a page's recent trend is down, while the operational question is which pages should be reviewed first.



I start with Logistic Regression because it is a readable linear benchmark, then compare a shallow Decision Tree and a constrained Random Forest for non-linear relationships. Model probabilities become ranking scores, so precision at 20, 50, and 100 is primary; average precision and balanced accuracy describe broader behavior. `trend_direction`, `trend_pct`, and recent-window fields that directly construct or overlap the label are excluded. IDs only define the grouped split. This is decision support, not a causal estimate of refresh impact.

In [6]:
from pathlib import Path

import json

import platform



import numpy as np

import pandas as pd

import sklearn

from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier

from sklearn.impute import SimpleImputer

from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import average_precision_score, balanced_accuracy_score

from sklearn.model_selection import GroupShuffleSplit

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.tree import DecisionTreeClassifier



SEED = 42

TOP_KS = (20, 50, 100)



# Prefer the checked-out starter CSV; hosted notebooks can use the public fallback.

candidates = [Path.cwd(), *Path.cwd().parents]

local_repo = Path("/Users/thany/Documents/Development/content-refresh-prioritizer")

if local_repo not in candidates:

    candidates.append(local_repo)

root = next(

    (

        candidate

        for candidate in candidates

        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists()

    ),

    None,

)



if root is not None:

    DATA_SOURCE = root / "data" / "raw" / "content_refresh_anonymized.csv"

    RESULTS_PATH = root / "work" / "outputs" / "w05_model_results.json"

else:

    DATA_SOURCE = (

        "https://raw.githubusercontent.com/flyrank-bih/"

        "flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

    )

    RESULTS_PATH = Path.cwd() / "w05_model_results.json"



df = pd.read_csv(DATA_SOURCE)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)



CONTEXT = ["content_id", "client_id"]

LABEL_OR_SOURCE = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]

EXCLUDED = [

    "provider_used", "model_used", "clicks_last_30d", "sessions_last_30d",

    "clicks_prev_30d", "sessions_prev_30d",

]

FEATURES = [

    column for column in df.columns

    if column not in CONTEXT + LABEL_OR_SOURCE + EXCLUDED + ["is_declining_label"]

]

NUMERIC_FEATURES = [column for column in FEATURES if pd.api.types.is_numeric_dtype(df[column])]

CATEGORICAL_FEATURES = [column for column in FEATURES if column not in NUMERIC_FEATURES]



forbidden = set(LABEL_OR_SOURCE + EXCLUDED + CONTEXT + ["is_declining_label"])

assert not (set(FEATURES) & forbidden), "Leakage or identifier entered the feature set."

assert df["content_id"].is_unique and df["client_id"].nunique() == 32



def precision_at_k(labels, scores, k):

    k = min(k, len(labels))

    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")[:k]

    return float(np.asarray(labels)[order].mean())



def baseline_scores(frame):

    impressions = frame["impressions_90d"].fillna(0)

    position = frame["avg_position"].fillna(0)

    ctr = frame["ctr"].fillna(0)

    word_count = frame["word_count"].fillna(0)

    update_age = frame["days_since_last_update"].fillna(0)

    content_age = frame["content_age_days"].fillna(0)

    problem_score = (

        2 * ((update_age >= 180) & (impressions >= 500)).astype(int)

        + 2 * ((impressions >= 500) & position.between(0, 20, inclusive="right") & (position > 0) & (ctr < 0.5)).astype(int)

        + ((word_count > 0) & (word_count < 1200) & (impressions >= 250)).astype(int)

        + ((position > 0) & (position <= 10) & (content_age >= 180)).astype(int)

    )

    visibility = impressions.rank(pct=True)

    return np.where(problem_score > 0, problem_score + visibility, 0.0)



source_label = "local CSV" if root is not None else "public starter CSV"

print(f"Loaded {len(df):,} pages from {df['client_id'].nunique()} pseudonymous clients ({source_label}).")

print(f"Target base rate: {df['is_declining_label'].mean():.3f} | safe features: {len(FEATURES)}")

print(f"Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

Loaded 30,000 pages from 32 pseudonymous clients (public starter CSV).
Target base rate: 0.542 | safe features: 32
Python 3.13.15 | pandas 2.2.3 | scikit-learn 1.6.1


## 2. Split design



Pages from the same client can share measurement and content patterns, so a random row split would make the test set too familiar. A deterministic `GroupShuffleSplit` holds out about 20% of clients and keeps every page from a client on one side. The code also verifies both classes remain in each partition. This tests transfer to unseen pseudonymous clients; it is not a time-forward claim because the starter release is a cross-sectional snapshot.

In [10]:
groups = df["client_id"]

labels = df["is_declining_label"]



for attempt in range(100):

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED + attempt)

    train_idx, test_idx = next(splitter.split(df, labels, groups))

    if labels.iloc[train_idx].nunique() == 2 and labels.iloc[test_idx].nunique() == 2:

        break

else:

    raise RuntimeError("Could not create a grouped split with both classes in each partition.")



train_df = df.iloc[train_idx].copy()

test_df = df.iloc[test_idx].copy()

assert set(train_df["client_id"]).isdisjoint(test_df["client_id"])



split_summary = pd.DataFrame(

    {

        "rows": [len(train_df), len(test_df)],

        "clients": [train_df["client_id"].nunique(), test_df["client_id"].nunique()],

        "decline_rate": [train_df["is_declining_label"].mean(), test_df["is_declining_label"].mean()],

    },

    index=["train", "held_out_test"],

)

split_summary

,rows,clients,decline_rate
train,23837,25,0.550111
held_out_test,6163,7,0.510952


## 3. Train + compare with the Week-4 baseline



Every row below uses the same held-out clients and labels. Median imputation includes explicit missing-value indicators for numeric fields; categorical missingness receives its own level. This preserves informative missingness without pretending that missing values are measured zeros. The selected model is the one with the best held-out precision@50, using average precision only as a tie-breaker.

In [11]:
def make_preprocessor():

    numeric_pipeline = Pipeline(

        [

            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),

            ("scaler", StandardScaler()),

        ]

    )

    categorical_pipeline = Pipeline(

        [

            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),

            ("one_hot", OneHotEncoder(handle_unknown="ignore")),

        ]

    )

    return ColumnTransformer(

        [

            ("numeric", numeric_pipeline, NUMERIC_FEATURES),

            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),

        ]

    )



models = {

    "logistic_regression": LogisticRegression(

        class_weight="balanced", max_iter=2000, random_state=SEED

    ),

    "decision_tree": DecisionTreeClassifier(

        class_weight="balanced", max_depth=3, min_samples_leaf=75, random_state=SEED

    ),

    "random_forest": RandomForestClassifier(

        n_estimators=300, class_weight="balanced_subsample", max_depth=10,

        min_samples_leaf=25, n_jobs=-1, random_state=SEED,

    ),

}



def evaluate(name, y_true, scores):

    predicted = np.asarray(scores) >= 0.5

    row = {

        "method": name,

        "average_precision": average_precision_score(y_true, scores),

        "balanced_accuracy": balanced_accuracy_score(y_true, predicted),

    }

    row.update({f"precision_at_{k}": precision_at_k(y_true, scores, k) for k in TOP_KS})

    return row



X_train = train_df[FEATURES]

X_test = test_df[FEATURES]

y_train = train_df["is_declining_label"]

y_test = test_df["is_declining_label"]



all_baseline = baseline_scores(df)

rows = [evaluate("week_4_rule_baseline", y_test, all_baseline[test_idx])]

fitted_models = {}

test_scores = {}



for name, estimator in models.items():

    pipeline = Pipeline([("preprocess", make_preprocessor()), ("model", estimator)])

    pipeline.fit(X_train, y_train)

    scores = pipeline.predict_proba(X_test)[:, 1]

    fitted_models[name] = pipeline

    test_scores[name] = scores

    rows.append(evaluate(name, y_test, scores))



comparison = pd.DataFrame(rows).set_index("method")

comparison.loc["held_out_base_rate"] = {

    "average_precision": y_test.mean(),

    "balanced_accuracy": 0.5,

    **{f"precision_at_{k}": y_test.mean() for k in TOP_KS},

}

model_names = list(models)

best_model_name = max(

    model_names,

    key=lambda name: (

        comparison.loc[name, "precision_at_50"],

        comparison.loc[name, "average_precision"],

    ),

)

print(f"Selected model: {best_model_name}")

comparison.round(3)

Selected model: logistic_regression


,average_precision,balanced_accuracy,precision_at_20,precision_at_50,precision_at_100
method,,,,,
week_4_rule_baseline,0.508,0.515,0.350,0.380,0.350
logistic_regression,0.575,0.558,0.650,0.720,0.690
decision_tree,0.568,0.573,0.700,0.660,0.620
random_forest,0.585,0.578,0.550,0.540,0.570
held_out_base_rate,0.511,0.500,0.511,0.511,0.511


## 4. Errors and interpretation



Permutation importance measures how much held-out average precision falls when one raw feature is shuffled. This is directional model reliance, not causality. The error table checks whether mistakes concentrate by content type, and the three cases show the most confident wrong predictions without exposing client identities, URLs, or queries. A suspiciously dominant feature would trigger a return to the leakage audit before deployment.

In [12]:
best_model = fitted_models[best_model_name]

best_scores = test_scores[best_model_name]



# Use a fixed held-out subsample to keep permutation importance quick and reproducible.

importance_size = min(3000, len(X_test))

importance_sample = X_test.sample(importance_size, random_state=SEED)

importance_labels = y_test.loc[importance_sample.index]

importance = permutation_importance(

    best_model,

    importance_sample,

    importance_labels,

    scoring="average_precision",

    n_repeats=3,

    random_state=SEED,

    n_jobs=-1,

)

importance_table = (

    pd.DataFrame(

        {

            "feature": FEATURES,

            "importance_mean": importance.importances_mean,

            "importance_sd": importance.importances_std,

        }

    )

    .sort_values("importance_mean", ascending=False)

    .head(10)

    .reset_index(drop=True)

)



error_frame = test_df[["content_type", "is_declining_label"]].copy()

error_frame["decline_probability"] = best_scores

error_frame["predicted_label"] = (best_scores >= 0.5).astype(int)

error_frame["wrong"] = error_frame["predicted_label"] != error_frame["is_declining_label"]

errors_by_type = (

    error_frame.groupby("content_type", dropna=False)

    .agg(rows=("wrong", "size"), errors=("wrong", "sum"), error_rate=("wrong", "mean"))

    .sort_values(["error_rate", "rows"], ascending=[False, False])

)



case_columns = [

    "content_type", "impressions_90d", "avg_position", "ctr",

    "days_since_last_update", "is_declining_label",

]

wrong_cases = test_df.loc[error_frame["wrong"], case_columns].copy()

wrong_cases["decline_probability"] = error_frame.loc[error_frame["wrong"], "decline_probability"]

wrong_cases["confidence"] = (wrong_cases["decline_probability"] - 0.5).abs()

wrong_cases = wrong_cases.sort_values("confidence", ascending=False).head(3).reset_index(drop=True)

wrong_cases.index = wrong_cases.index + 1

wrong_cases.index.name = "case"



result_payload = {

    "seed": SEED,

    "split": {

        "strategy": "group_shuffle_split_by_client",

        "train_rows": int(len(train_df)),

        "test_rows": int(len(test_df)),

        "train_clients": int(train_df["client_id"].nunique()),

        "test_clients": int(test_df["client_id"].nunique()),

        "test_base_rate": float(y_test.mean()),

    },

    "selected_model": best_model_name,

    "metrics": {

        name: {metric: float(value) for metric, value in row.items()}

        for name, row in comparison.drop(index="held_out_base_rate").to_dict(orient="index").items()

    },

    "top_permutation_features": importance_table.head(5).to_dict(orient="records"),

    "versions": {

        "python": platform.python_version(),

        "pandas": pd.__version__,

        "scikit_learn": sklearn.__version__,

    },

}

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

RESULTS_PATH.write_text(json.dumps(result_payload, indent=2))



print("Top held-out permutation features:")

display(importance_table.round(4))

print("Error concentration by content type:")

display(errors_by_type.round(3))

print("Three high-confidence wrong cases:")

display(wrong_cases.drop(columns="confidence").round(3))

print(f"Saved reproducible metrics to {RESULTS_PATH}")

Top held-out permutation features:


,feature,importance_mean,importance_sd
0,days_with_impressions,0.0512,0.0068
1,avg_position,0.0347,0.0036
2,content_age_days,0.0344,0.0036
3,days_with_sessions,0.0335,0.0038
4,users_90d,0.0208,0.0021
5,scroll_rate,0.0109,0.0019
6,word_count,0.0088,0.0024
7,freshness_tier,0.0059,0.0011
8,age_tier_order,0.0058,0.0013
9,impression_tier,0.0044,0.0017


Error concentration by content type:


,rows,errors,error_rate
content_type,,,
keyword article,6163,2715,0.441


Three high-confidence wrong cases:


,content_type,impressions_90d,avg_position,ctr,days_since_last_update,is_declining_label,decline_probability
case,,,,,,,
1,keyword article,3,2.0,0.0,20,1,0.066
2,keyword article,916,78.6,0.0,22,1,0.067
3,keyword article,643,76.0,0.0,20,1,0.069


Saved reproducible metrics to /content/w05_model_results.json


## Self-check



- [x] Every analysis section contains both reasoning and supporting code

- [x] All code cells executed in order with no errors

- [x] No client names, URLs, or private queries are displayed

- [x] Claims use observed, measured, directional, and decision-support language

- [ ] Commit under `work/notebooks/`, then submit the repository URL on the card



The notebook prefers the repository CSV and falls back to the public starter CSV in hosted kernels. Hosted runs write the metrics receipt to the kernel working directory; local repository runs write it to `work/outputs/w05_model_results.json`.